# PCG VGG16

A/P 位置 PCG，辨識 N / AR / PR。三個模型共用同一組病人切分，模型訓練使用各自 seed。


In [ ]:
# 1. Seed 與病人層級切分設定（最先執行）
SPLIT_SEED = 3894
MODEL_SEED = 9127
VAL_SIZE = 0.15
TEST_SIZE = 0.15

import os, random
import numpy as np
import torch

os.environ["PYTHONHASHSEED"] = str(MODEL_SEED)
random.seed(MODEL_SEED)
np.random.seed(MODEL_SEED)
torch.manual_seed(MODEL_SEED)
torch.cuda.manual_seed_all(MODEL_SEED)
print("shared patient split seed:", SPLIT_SEED)
print("model seed:", MODEL_SEED)
print("病人切分在 window 建立前完成；A/P 共用相同 train/validation/test 病人。")


In [ ]:
# 2. 路徑與模型參數
from pathlib import Path
import subprocess

PYTHON = Path(r"C:\Users\USER\AppData\Local\Programs\Python\Python311\python.exe")
PROJECT = Path(r"C:\Users\USER\Desktop\PECO_v2")
DATA_ROOT = Path("D:/Dataset_train")
OUTPUT_ROOT = Path("D:/PECO_PCG_runs/notebook_1107")
RUN_NAME = "pcg_ap_vgg16_notebook_seed9127"
RUN_ROOT = OUTPUT_ROOT / RUN_NAME
MODEL = "vgg16"

# 三個 PCG notebook 共用這份由 split seed 3894 產生的病人清單；CSV 不放在本 notebook 資料夾。
PATIENT_SPLIT = Path("D:/PECO_PCG_runs/pcg_valve_tms_weighted_t1_m2_s3_seed_3894/A/vgg16/file_metadata.csv")
assert PATIENT_SPLIT.exists(), PATIENT_SPLIT


In [ ]:
# 3. 最終分類決策（直接使用已確定的值）
PATIENT_AGGREGATION = "top3"
FINAL_THRESHOLDS = {"N": 0.375, "AR": 0.65, "PR": 0.825}
print("patient aggregation:", PATIENT_AGGREGATION)
print("final thresholds:", FINAL_THRESHOLDS)


## 前處理

PCG 重採樣為 1000 Hz；每段 2 秒。N stride 2 秒，AR/PR stride 1 秒。每段做 z-score，再轉成 48-band log-mel（FFT 128、hop 32）；第二通道為位置編碼 A=-1、P=+1。


In [ ]:
# 4. 訓練
cmd = [
    str(PYTHON), str(PROJECT / "train_pcg_valve_multilabel.py"),
    "--data-root", str(DATA_ROOT), "--output-dir", str(OUTPUT_ROOT),
    "--locations", "A", "P", "--classes", "N", "AR", "PR",
    "--combine-locations", "--location-group-name", "AP",
    "--patient-split-csv", str(PATIENT_SPLIT), "--models", MODEL,
    "--target-sr", "1000", "--feature-mode", "logmel",
    "--mel-n-fft", "128", "--mel-hop-length", "32", "--mel-n-mels", "48",
    "--logmel-width-multiplier", "0.5",
    "--window-sec", "2", "--normal-stride-sec", "2", "--positive-stride-sec", "1",
    "--positive-grades", "T", "M", "S",
    "--grade-weight-t", "1", "--grade-weight-m", "2", "--grade-weight-s", "3",
    "--val-size", str(VAL_SIZE), "--test-size", str(TEST_SIZE),
    "--seed", str(MODEL_SEED), "--epochs", "40", "--batch-size", "32",
    "--lr", "0.0001", "--weight-decay", "0.002", "--max-pos-weight", "20",
    "--select-level", "patient", "--select-metric", "macro_f1",
    "--early-stop-patience", "10", "--weighted-sampler", "--sampler-mode", "patient_class",
    "--cache-waveforms", "--augment", "--augment-jitter-sec", "0.1",
    "--augment-noise-std", "0.02", "--augment-polarity-flip",
    "--mixup-alpha", "0.3", "--use-location-channel",
    "--num-workers", "0", "--device", "cuda", "--amp",
    "--run-name", RUN_NAME, "--no-progress",
]
cmd.append("--resnet-imagenet-pretrained")
subprocess.run(cmd, check=True)


In [ ]:
# 5. 顯示實際病人數；split 欄位在建立任何 window 前已固定
import json
model_dir = RUN_ROOT / "AP" / MODEL
summary = json.loads((model_dir / "split_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))
